**INF201 - Deliverable 2: Kachina Aschenberg Øwre & Seline Heggelund Knutsen**

Link to our github repository: https://github.com/kacoew/INF201-CAs


## Introduction
This notebook builds a data pipeline for the Campus Coffee Shop chain. It finds all
register log files, extracts valid sales lines, cleans price errors and creates
summary reports (Tasks 1–4 build the functions, Task 5 combines them).

**Before running:** generate the input data with `python generate_data_for_d2.py`.
The `data/` folder is not in the GitHub repository.
All output files are saved in the `output/` folder.

In [1]:
import json # for reading JSON files
import re # for regular expressions
from pathlib import Path # for file paths
import numpy as np # for numerical operations
import pandas as pd # for data handling and analysis

DATA_DIR = Path("data")            # relative path -> works on both laptops
LOG_ROOT = DATA_DIR / "2026"
CONFIG_FILE = DATA_DIR / "store_config.json"
OUTPUT_DIR = Path("output")
EPSILON = 1e-6                     # Task 3.2: avoids division by zero

## Task 1.1 – Robust file reader
* Use `pathlib.Path`.
* Handle both `UTF-8` and `ISO-8859-1` without hard-coding which file has which.
* Return `(file_text_content, encoding_used)`.

**Our solution:** We do not know which encoding a file has, so the function tries to read it as UTF-8. 
If that fails with a `UnicodeDecodeError`, it reads the file again as ISO-8859-1 (Latin-1). 
The order matters, Latin-1 accepts every possible byte and never fails, so if we tried it first, UTF-8 files with special characters would be read incorrectly without any error. 
We only catch `UnicodeDecodeError`, so other problems (e.g. a wrong file path) still raise an error.

In [2]:
# Task 1.1: read_text_file

def read_text_file(file_path: str | Path) -> tuple[str, str]:
    """Read a text file whose encoding is either UTF-8 or ISO-8859-1.

    Returns a tuple (file_text_content, encoding_used).
    """
    path = Path(file_path)
    try:
        return path.read_text(encoding="utf-8"), "utf-8" 
    except UnicodeDecodeError:  # If UTF-8 raises UnicodeDecodeError, read as ISO-8859-1 instead
        # Latin-1 maps every byte to a character, so this fallback cannot fail.
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1" 
    

In [3]:
# Test both branches (the real log files are plain ASCII, so they only hit UTF-8).
test_dir = OUTPUT_DIR / "encoding_test"
test_dir.mkdir(parents=True, exist_ok=True)

(test_dir / "latin1.txt").write_text("Café Crème", encoding="iso-8859-1")
(test_dir / "utf8.txt").write_text("Café Crème", encoding="utf-8")

print(read_text_file(test_dir / "latin1.txt"))
print(read_text_file(test_dir / "utf8.txt"))

('Café Crème', 'iso-8859-1')
('Café Crème', 'utf-8')


**What the test shows:** The generated log files only contain ASCII characters, so they are
always read with UTF-8 and never test the fallback. We therefore wrote "Café Crème" to two test files with different encodings. Both are read correctly, and the function reports the correct encoding for each, which shows that both branches work.

## Task 1.2 – Log parser with regular expressions
Valid line: `[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)`

Noise to ignore, e.g. `--- PRINTER PAPER LOW ---`, `=== NETWORK SYNC OK ===`, `[DEBUG] BARCODE SCANNER REBOOTED`.

Capture 6 fields: `timestamp`, `register_id`, `item`, `price` (float), `currency`, `status`.
Return a list of dicts.

**Our solution:** We use one regular expression with a named group for each of the six fields:

    [2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
     └──── timestamp ───┘  └─┬──┘       └─┬─┘       └┬─┘ └┬┘  └──┬──┘
                        register_id     item      price currency status

In [4]:
# SALES_LINE is a regex pattern that matches a sales line in the log file. 
# List of strings:
# - timestamp: format YYYY-MM-DD HH:MM:SS
# - register_id: word characters (letters, digits, and underscores)
# - item: word characters (letters, digits, and underscores)
# - price: digits followed by a decimal point and more digits
# - currency: three uppercase letters
# - status: either 'SUCCESS', 'REFUND', or 'CANCELLED' 

SALES_LINE = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>\w+)\] "
    r"item=(?P<item>\w+) "
    r"price=(?P<price>\d+\.\d+) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$",
    re.MULTILINE, # MULTILINE flag allows the regex to match each line in the text separately
)

def parse_log(text: str) -> list[dict]:
    """Return one dict per valid sales line in ``text``; noise lines are skipped.
    Each dict has keys: timestamp, register_id, item, price, currency, status.
    """
    records = []
    for match in SALES_LINE.finditer(text): 
        record = match.groupdict()
        record["price"] = float(record["price"])
        records.append(record)
    return records



In [5]:
sample = """[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
--- PRINTER PAPER LOW ---
[DEBUG] BARCODE SCANNER REBOOTED
=== NETWORK SYNC OK ==="""

result = parse_log(sample)
print(len(result))   # expected: 1
print(result)

1
[{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}]


**What the test shows:** Of the four test lines, only the valid sales line is returned, and
its price is the float `4.5`. The line `[DEBUG] BARCODE SCANNER REBOOTED` is an important
test case, because it starts with `[` just like a sales line. It is still skipped, because
the full pattern does not match.

## Task 2.1 – Recursive utility functions
Use `Path.iterdir()` and explicit recursion. **Not allowed:** `glob`, `rglob`, `os.walk`.

**Our solution:** The function calls itself on every subfolder; we sort the result because
`iterdir()` gives no fixed order.

In [6]:
def find_log_files(directory: str | Path, extension: str = ".log") -> list[Path]:
    """Find all log files in nested directories.
    
    Recursively collect every file with ``extension`` below ``directory``.
    
    Return a sorted list of Path objects.
    """
    directory = Path(directory)
    found: list[Path] = []
    for child in directory.iterdir(): # iterate over all files and directories in the given directory
        if child.is_dir(): # if the child is a directory, recursively call find_log_files on it
            found.extend(find_log_files(child, extension))
        elif child.is_file() and child.suffix == extension: # if the child is a file and has the given extension, add it to the list
            found.append(child)
    return sorted(found)   # fixed order -> reproducible output

In [7]:
# Find all log files in the LOG_ROOT directory and its subdirectories.
log_files = find_log_files(LOG_ROOT)
print(len(log_files)) 
print(log_files[0])

9
data\2026\library_cafe\POS_01\register_sales.log


**What the test shows:** The function finds 9 log files: 3 stores × 3 registers, with one
`register_sales.log` in each register folder. This matches the folder structure created by
`generate_data_for_d2.py`.

## Task 2.2 – Active register IDs from nested JSON
Traverse any combination of nested dicts and lists. Return `set[str]` of every register `id` where `"active": true`.

**Our solution:** Text, numbers and booleans are the base case; we return a `set`, so an ID
that appears several times is only stored once.

In [8]:
def find_active_registers(node) -> set[str]:
    """Recursively collect the IDs of all active registers in a nested config.

    ``node`` can be any mix of dicts and lists (e.g. the loaded
    store_config.json). A dict counts as an active register if it has an
    "id" key and "active" is True.

    Returns a set of register ID strings, e.g. {"POS_01", "POS_02"}.
    """
    active: set[str] = set()
    if isinstance(node, dict): 
        if node.get("active") is True and "id" in node:
            active.add(node["id"])
        for value in node.values():
            active |= find_active_registers(value)
    elif isinstance(node, list):
        for element in node:
            active |= find_active_registers(element)
    return active

In [9]:
# Test the function with the store_config.json file.
config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
print(find_active_registers(config))   # expected {'POS_01', 'POS_02', 'POS_03'}

{'POS_01', 'POS_02', 'POS_03'}


**What the test shows:** The active registers are POS_01, POS_02 and POS_03. POS_04 is not
included, because it has `"active": false`. All register IDs that appear in the log files
(POS_01–POS_03) are active, so filtering on active registers would not remove any transactions.
Note that the config only lists two stores (no Student Union), and the same ID (e.g. POS_01)
is used in several stores, so a register ID alone does not identify one specific register.

## Task 3.1 – Remove price outliers
Convert to a 1D float array, flag values with $\lvert x - \mu \rvert > 3\sigma$, replace them with the median of the valid prices.

**Our solution:** The prices are first converted to a 1D float NumPy array. We use
`np.array`, which makes a copy, so the original data is not changed. 
We find all outliers at once with a boolean mask and replace them with the median of the
remaining (valid) prices.

In [10]:
def remove_price_outliers(prices) -> np.ndarray:
    """Replace prices with |x - mean| > 3 * std by the median of the other prices.

    Returns a new 1D float array; the input is not changed.
    """
    values = np.array(prices, dtype=float).ravel()  # np.array makes a COPY - pandas 3 may give a read-only array otherwise
    mu, sigma = values.mean(), values.std()
    is_outlier = np.abs(values - mu) > 3 * sigma
    values[is_outlier] = np.median(values[~is_outlier])
    return values

In [11]:
# Test with MANY normal values + one glitch (with only 2-3 values, 
# sigma is so large that 999 is NOT > 3 sigma):

test = [4.0, 4.5, 5.0, 3.5, 4.2, 4.8, 3.9, 4.1, 4.6, 4.4, 999.0]
print(remove_price_outliers(test))

[4.  4.5 5.  3.5 4.2 4.8 3.9 4.1 4.6 4.4 4.3]


**What the test shows:** In the test list, 999.0 is replaced by 4.3, the median of the ten
normal prices. On our real data (720 prices), the mean is 45.86 and the standard deviation
is 198.75, both strongly inflated by the glitches. The limit is therefore about 642, but
999.00 is still far above it, so all 30 glitch values are detected. Afterwards the highest
price is 8.00.

**Limitation:** The 3σ rule only works because 999 is so extreme. With very few values
(e.g. `[4, 5, 999]`) the standard deviation becomes so large that 999 is not detected, and a
smaller glitch (e.g. 15.00) would not be caught. The median (3.97) is also calculated across
all items and currencies, so a glitched Sandwich price gets roughly the price of an Espresso.

## Task 3.2 – compute_refund_ratio
$$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$

**Our solution:** The function converts both inputs to float arrays and divides them in one
NumPy operation, which works element by element, so no loop is needed. A small number
$\epsilon = 10^{-6}$ is added to the sales count, so a count of 0 does not cause a division by zero.

In [12]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Return refund_count / (sales_count + EPSILON), element by element.

    EPSILON = 1e-6 avoids division by zero.
    """
    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)
    return refunds / (sales + EPSILON)     # no loop needed



In [13]:
print(compute_refund_ratio(np.array([10, 0]), np.array([2, 0])))


[0.19999998 0.        ]


**What the test shows:** 2 refunds out of 10 sales gives 0.19999998 instead of exactly 0.2,
because of ε. The difference is too small to matter. When both counts are 0, the result is 0
instead of an error or `NaN`, which shows that ε prevents division by zero.

## Task 4.1 – Build the DataFrame
List of dicts → DataFrame, `timestamp` as `datetime` index, sorted chronologically.

In [14]:


def create_dataframe(records: list[dict]) -> pd.DataFrame:
    """Convert transaction records into a DataFrame

    Convert timestamps to datetime, set them as the index,
    and sort transactions chronologically
    """
    # Convert the list of dictionaries into a DataFrame
    df = pd.DataFrame(records)

    # Convert timestamps to datetime objects
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Set timestamp as the index and sort chronologically
    df = df.set_index("timestamp")
    df = df.sort_index()

    return df


In [15]:
df = create_dataframe(result)

print(df)
print(df.index)

                    register_id   item  price currency   status
timestamp                                                      
2026-10-15 08:30:00      POS_01  Latte    4.5      EUR  SUCCESS
DatetimeIndex(['2026-10-15 08:30:00'], dtype='datetime64[ns]', name='timestamp', freq=None)


## Task 4.2 – Hourly revenue per item
Group by item, resample in 1-hour windows, sum the price.

In [18]:


def calculate_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the total price for each item in one-hour intervals."""

    hourly_revenue = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
        .rename("total_revenue")
    )

    return hourly_revenue


In [19]:
hourly = calculate_hourly_revenue(df)
print(hourly)

item   timestamp          
Latte  2026-10-15 08:00:00    4.5
Name: total_revenue, dtype: float64


## Task 4.3 – Pivot table: revenue by item × status

## Task 4.4 – Refund ratio per item
Uses `compute_refund_ratio` from Task 3.2.

*Our choice of "sales count":* TODO – write here whether you use only `SUCCESS` or all transactions, and why.

**Task 5**:

## Task 5 – Full workflow
1. Find log files + active register IDs → 2. read & parse → 3. DataFrame → 4. outlier cleaning →
5. NOK × 10 → 6. reports → 7. save the 4 output files.

## Discussion
TODO: results + reflection. Ideas: mixed currencies in "revenue",
REFUND/CANCELLED in the hourly sum, limits of the 3σ rule, why outliers are removed before NOK × 10,
the ASCII-only log files and your encoding test.

## Summary of Gen AI use
TODO